In [ ]:
import os 
import sys 
import json 
from pathlib import Path 

import torch 
import torch.nn as nn 
import numpy as np 
import matplotlib.pyplot as plt 
from torch.utils.data import DataLoader,TensorDataset 

project_root= Path.cwd()
while not(project_root /'requirements.txt').exists():
    if project_root= =project_root.parent :
        raise RuntimeError("Could not find project root(looked for requirements.txt)")
    project_root= project_root.parent 
sys.path.insert(0,str(project_root))

from src.models.classification import ClassificationModelHandler 
from src.utils.training import test_model,training_loop 

In [ ]:
device= torch.device('cuda'if torch.cuda.is_available()else 'cpu')
print(f"Using: {device}")

In [ ]:
DATASET= "BUSUCLM"
FEATURE_GROUP= "concatenated"
FEATURE_VARIANT= "basic_resnet"

num_classes= 2 
batch_size= 32 
num_epochs= 500 
learning_rate= 0.001 

offline_features_dir= project_root /'experiments'/'models'/'fusion'/'offline'/DATASET /FEATURE_GROUP /FEATURE_VARIANT 
labels_dir= project_root /'experiments'/'labels'/DATASET 
save_dir= project_root /'experiments'/'models'/'classification'/'offline'/DATASET /FEATURE_VARIANT 
results_dir= project_root /'experiments'/'results'/'offline'/DATASET /FEATURE_VARIANT 

train_features_file= offline_features_dir /'train_features_concat.npy'
val_features_file= offline_features_dir /'val_features_concat.npy'
test_features_file= offline_features_dir /'test_features_concat.npy'

train_labels_file= labels_dir /'train_labels.npy'
val_labels_file= labels_dir /'val_labels.npy'
test_labels_file= labels_dir /'test_labels.npy'

model_name= f"classification_{DATASET.lower()}_{FEATURE_VARIANT}"

os.makedirs(save_dir,exist_ok= True)
os.makedirs(results_dir,exist_ok= True)

print(f"Dataset: {DATASET}")
print(f"Feature group: {FEATURE_GROUP}")
print(f"Feature variant: {FEATURE_VARIANT}")
print(f"Features directory: {offline_features_dir}")
print(f"Labels directory: {labels_dir}")
print(f"Model directory: {save_dir}")
print(f"Results directory: {results_dir}")

In [ ]:
train_features= np.load(train_features_file)
val_features= np.load(val_features_file)
test_features= np.load(test_features_file)

train_labels= np.load(train_labels_file).squeeze().astype(np.int64)
val_labels= np.load(val_labels_file).squeeze().astype(np.int64)
test_labels= np.load(test_labels_file).squeeze().astype(np.int64)

print(f"Train features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Validation labels shape: {val_labels.shape}")
print(f"Test labels shape: {test_labels.shape}")
print(f"Feature dimension: {train_features.shape [1]}")

In [ ]:
def build_loader(features,labels,batch_size,shuffle= False):
    features_tensor= torch.FloatTensor(features)
    labels_tensor= torch.LongTensor(labels)
    dataset= TensorDataset(features_tensor,labels_tensor)
    return DataLoader(dataset,batch_size= batch_size,shuffle= shuffle)

val_loader= build_loader(val_features,val_labels,batch_size= batch_size,shuffle= False)
test_loader= build_loader(test_features,test_labels,batch_size= batch_size,shuffle= False)

print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
train_loader= build_loader(train_features,train_labels,batch_size= batch_size,shuffle= True)

print(f"Number of training batches: {len(train_loader)}")

In [ ]:
sequential_hyperparams= {
'input_dim':train_features.shape [1 ],
'learning_rate':learning_rate 
}

classifier= ClassificationModelHandler(
device= device,
num_classes= num_classes,
sequential_hyperparams= sequential_hyperparams 
)

print("Model:")
print(classifier)

In [ ]:
criterion= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(classifier.model.parameters(),lr= learning_rate)

history= training_loop(
model= classifier.model,
train_loader= train_loader,
val_loader= val_loader,
criterion= criterion,
optimizer= optimizer,
device= device,
num_epochs= num_epochs,
save_dir= str(save_dir),
model_name= model_name,
extract_features= False,
save_best_only= True,
save_all_epochs= False 
)

best_model_path= save_dir /f"best_{model_name}.pth"

print("\nTraining completed.")
print(f"Best model path: {best_model_path}")

In [ ]:
best_model_path= save_dir /f"best_{model_name}.pth"

if best_model_path.exists():
    classifier.model.load_state_dict(torch.load(best_model_path,map_location= device))
    print(f"Best model loaded from: {best_model_path}")
else :
    print("Warning: best model file not found. Using current model weights.")

classifier.model.eval()

In [ ]:
criterion= nn.CrossEntropyLoss()

hyperparams= {
'dataset':DATASET,
'feature_group':FEATURE_GROUP,
'feature_variant':FEATURE_VARIANT,
'num_classes':num_classes,
'batch_size':batch_size,
'num_epochs':num_epochs,
'learning_rate':learning_rate,
'input_dim':train_features.shape [1 ]
}

output_file= results_dir /'test_results.txt'
metrics_file= results_dir /'metrics_summary.json'

test_loss,accuracy,precision,recall,f1,auc= test_model(
model= classifier.model,
dataloader= test_loader,
criterion= criterion,
device= device,
hyperparams= hyperparams,
output_file= str(output_file),
extract_features= False,
compute_detailed_metrics= True 
)

summary= {
'dataset':DATASET,
'feature_group':FEATURE_GROUP,
'feature_variant':FEATURE_VARIANT,
'model_name':model_name,
'best_model_path':str(best_model_path),
'results_file':str(output_file),
'metrics':{
'test_loss':float(test_loss),
'accuracy':float(accuracy),
'precision':float(precision),
'recall':float(recall),
'f1_score':float(f1),
'auc':float(auc)
}
}

with open(metrics_file,'w')as f :
    json.dump(summary,f,indent= 4)

print(f"\n{'='*50}")
print("FINAL TEST RESULTS:")
print(f"{'='*50}")
print(f"Test Loss: {test_loss :.4f}")
print(f"Accuracy: {accuracy :.4f}")
print(f"Precision: {precision :.4f}")
print(f"Recall: {recall :.4f}")
print(f"F1-Score: {f1 :.4f}")
print(f"AUC: {auc :.4f}")
print(f"{'='*50}")
print(f"Detailed results saved at: {output_file}")
print(f"Metrics summary saved at: {metrics_file}")